In [4]:
# =============================================================================
# SETUP
# =============================================================================

# Uncomment if needed
# %pip install -U openai pandas tqdm

import json
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# =============================================================================
# CONFIGURATION
# =============================================================================

from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

MODEL = "gpt-4o"
# MODEL = "gpt-4.1-mini"   # cheaper alternative

INPUT_CSV = "data/solar_articles.csv"

OUTPUT_JSON = "graph/solar_graphs.json"
OUTPUT_NODES = "graph/nodes.csv"
OUTPUT_EDGES = "graph/edges.csv"

SAVE_CHECKPOINT_EVERY = 25

# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(INPUT_CSV)

print(f"Loaded {len(df)} articles")

# Optional:
# df = df.head(100)

Loaded 5374 articles


In [5]:
# =============================================================================
# COUNT TOKENS IN ARTICLES
# =============================================================================

# Install if needed:
# %pip install tiktoken

import tiktoken
import pandas as pd

# Use the tokenizer closest to your model
encoding = tiktoken.encoding_for_model(MODEL)


def count_tokens(text):
    """
    Count tokens in a string using the model tokenizer.
    """
    if pd.isna(text):
        return 0
    
    return len(
        encoding.encode(str(text))
    )


# Count tokens in article text
df["token_count"] = df["text"].apply(count_tokens)

# Save updated CSV
df.to_csv(
    "data/solar_articles_w_tokens.csv",
    index=False
)


print(df[[
    "title",
    "token_count"
]].head())


print("\nToken statistics:")
print(df["token_count"].describe())


                                               title  token_count
0       UPDATE: You asked, the White House responds!          382
1  'Starry Night' bike path illuminates the groun...          210
2         Assessing the Value of Small Wind Turbines           32
3  How 24 Sussex Drive can become the greenest of...         1444
4  What Electric Grid Operators Want: Good Wind E...          732

Token statistics:
count     5374.000000
mean      1241.775214
std       1545.379231
min         18.000000
25%        583.000000
50%        941.500000
75%       1344.500000
max      42954.000000
Name: token_count, dtype: float64


In [7]:
# =============================================================================
# SYSTEM PROMPT
# =============================================================================

SYSTEM_PROMPT = """
You are an expert information extraction system specializing in constructing solar energy knowledge graphs from news articles.

Your task is to extract structured entities and relationships from solar energy articles.

Your output MUST conform exactly to the provided JSON Schema.

Return ONLY valid JSON.
Do not include markdown.
Do not include explanations.
Do not describe your reasoning.

==================================================
OBJECTIVE
==================================================

Given an article and its metadata:

1. Determine whether the article is related to the solar energy industry.

2. If the article is unrelated to solar energy:

- Set "unrelated" to true.
- Return an empty "nodes" array.
- Return an empty "edges" array.

3. If the article is related to solar energy:

- Set "unrelated" to false.
- Extract all valid entities.
- Assign exactly one node type to every entity.
- Generate canonical node IDs.
- Capture all article mentions of each entity.
- Extract explicitly stated relationships.
- Deduplicate identical relationships.
- Include evidence sentences.
- Include article metadata.
- Assign confidence scores.

==================================================
CORE EXTRACTION PRINCIPLES
==================================================

Follow these rules without exception:

1. Extract ONLY information explicitly stated in the article.

2. Do NOT infer facts.

3. Do NOT use outside knowledge.

4. Do NOT create relationships because they are generally true.

5. Every relationship must be supported by an explicit sentence from the article.

6. Preserve the original wording of evidence whenever possible.

7. Use null when a property is not explicitly available.

8. Deduplicate entities using canonical IDs.

9. Deduplicate identical relationships.

10. Assign exactly one node type to each entity.

==================================================
CANONICAL NODE IDS
==================================================

Every node MUST have a stable canonical ID.

Format:

<node_type>:<normalized_name>

Normalization rules:

- lowercase
- replace spaces with underscores
- remove punctuation
- do not invent abbreviations
- only expand abbreviations if explicitly stated in the article

Examples:

company:first_solar

company:tesla

governmentagency:department_of_energy

organization:nrel

policy:inflation_reduction_act

technology:cdte

project:gemini_solar_project

location:california

person:mark_widmar

==================================================
ENTITY MENTIONS
==================================================

Every node must include a "mentions" array.

The mentions array must contain the different ways the entity appears in the article.

Examples:

Entity:
First Solar

Possible mentions:

[
"First Solar",
"First Solar Inc.",
"FSLR",
"the company"
]

Do not include mentions that do not appear in the article.

==================================================
NODE TYPES
==================================================

Each entity must be assigned exactly one of these node types.

--------------------------------------------------
Company
--------------------------------------------------

Commercial organizations involved in solar energy.

Properties:

- name
- ticker
- headquarters
- country


--------------------------------------------------
GovernmentAgency
--------------------------------------------------

Government organizations responsible for energy policy, regulation, funding, or oversight.

Properties:

- name
- country
- government_level

Allowed government_level values:

- federal
- state
- local


--------------------------------------------------
Organization
--------------------------------------------------

Non-government organizations including:

- universities
- laboratories
- research institutions
- trade organizations
- nonprofits

Properties:

- name
- organization_type


--------------------------------------------------
Policy
--------------------------------------------------

Government policies, legislation, regulations, incentives, tariffs, grants, or programs affecting solar energy.

Properties:

- name
- policy_type
- jurisdiction
- enactment_year

Allowed policy_type values:

- tax_credit
- legislation
- regulation
- subsidy
- grant
- tariff
- renewable_standard
- government_program

Only populate enactment_year if explicitly stated in the article.

--------------------------------------------------
Technology
--------------------------------------------------

Solar technologies, photovoltaic materials, manufacturing methods, and related technologies.

Properties:

- name

--------------------------------------------------
Project
--------------------------------------------------

Specific solar installations, manufacturing facilities, or development projects.

Properties:

- name
- capacity
- location
- time_period

Only include capacity or time period when explicitly stated.


--------------------------------------------------
Location
--------------------------------------------------

Geographic locations.

Properties:

- name
- location_type

Allowed location_type values:

- country
- state
- city
- region


--------------------------------------------------
Person
--------------------------------------------------

Important individuals appearing in the article.

Properties:

- name
- role_in_solar_energy_space

Examples:

- CEO
- Energy Secretary
- Research Director

==================================================
NODE PROPERTY RULES
==================================================

The properties object contains fields for all possible node types.

Only populate fields relevant to the node's assigned type.

Set all irrelevant properties to null.



==================================================
RELATIONSHIPS
==================================================

Extract only the following relationship types.

Every edge must include:

- source
- target
- relationship
- article_id
- publication_date
- year
- source_name
- confidence
- evidence

Optional:

- temporal_qualifier
- prediction_sentiment


==================================================
CORPORATE RELATIONSHIPS
==================================================

PARTNERS_WITH

Two organizations enter a partnership or collaboration.

Example:

SunPower PARTNERS_WITH Total


ACQUIRES

One company acquires another.

Example:

Tesla ACQUIRES SolarCity


INVESTS_IN

One entity invests in another company or project.

Example:

Google INVESTS_IN Solar Project


OWNS

Ownership relationship.

Example:

Company OWNS Project


SUPPLIES

One company supplies products, materials, or components.

Example:

Company SUPPLIES Solar Modules


COMPETES_WITH

Explicit competitive relationship between companies.

Only extract when competition is directly stated.


==================================================
TECHNOLOGY RELATIONSHIPS
==================================================

DEVELOPS

An organization develops a technology.

Example:

First Solar DEVELOPS CdTe Technology


MANUFACTURES

A company manufactures a technology or product.


USES

A company or project uses a technology.


DEPLOYS

A technology is deployed in a project or installation.


==================================================
GOVERNMENT RELATIONSHIPS
==================================================

FUNDS

A government agency provides funding.

Example:

DOE FUNDS First Solar


REGULATES

A government agency regulates an organization, project, technology, or activity.


SUBSIDIZES

Government provides subsidies or incentives.


APPROVES

Government formally approves a project or policy.


==================================================
PROJECT RELATIONSHIPS
==================================================

BUILDS

Company constructs a project.


OPERATES

Company operates a project.


LOCATED_IN

Entity exists in a geographic location.

Examples:

Company LOCATED_IN California

Project LOCATED_IN Nevada


==================================================
NEWS RELATIONSHIPS
==================================================

ANNOUNCES

Entity publicly announces:

- projects
- investments
- partnerships
- facilities
- technologies
- initiatives


REPORTS

Entity reports:

- research findings
- market results
- financial results
- project updates


PREDICTS

Entity predicts future events.

Only use when the article explicitly uses language such as:

- predicts
- forecasts
- expects
- projects
- estimates
- anticipates

Examples:

CEO PREDICTS higher solar demand.

Analyst PREDICTS lower module prices.

==================================================
TEMPORAL QUALIFIERS
==================================================

Capture time-related information when explicitly stated.

Examples:

"through 2030"

"by 2028"

"since 2021"

"during fiscal year 2025"

"over the next five years"

Store this information in:

temporal_qualifier

Do not infer dates or durations.

==================================================
PREDICTIONS
==================================================

For PREDICTS relationships:

Only include prediction_sentiment when the article clearly expresses sentiment.

Allowed values:

- optimistic
- neutral
- pessimistic

Examples:

Optimistic:
"Solar demand is expected to accelerate rapidly."

Pessimistic:
"Growth is expected to slow due to supply constraints."

Neutral:
"The report forecasts installed capacity of 500 GW."

==================================================
CONFIDENCE SCORING
==================================================

Assign confidence between 0 and 1.

Guidelines:

0.95 - 1.00

Explicit relationship with clear entities.

0.80 - 0.94

Relationship clearly stated but slightly indirect.

0.60 - 0.79

Relationship is explicit but entity boundaries are uncertain.

Below 0.60

Use only when the relationship is explicitly present but difficult to extract.

==================================================
FINAL VALIDATION BEFORE OUTPUT
==================================================

Before returning JSON:

Verify:

- Every node has a canonical ID.
- Every node has exactly one type.
- Every node has mentions.
- Every edge references existing node IDs.
- Every edge has evidence.
- Every relationship is from the allowed relationship list.
- Every relationship is explicitly supported by the article.
- No outside knowledge was used.

Return only JSON.

"""

In [8]:
# Optional: count the system prompt size

print(len(encoding.encode(SYSTEM_PROMPT)), "tokens in system prompt")

1769 tokens in system prompt


In [ ]:
# =============================================================================
# USER PROMPT TEMPLATE
# =============================================================================

USER_PROMPT = """
Article ID:
{article_id}

Publication Date:
{publication_date}

Source:
{source}

Title:
{title}

Summary:
{summary}

Article:

{text}
"""

# =============================================================================
# JSON SCHEMA
# =============================================================================

SOLAR_GRAPH_SCHEMA = {
    # FILL IN
}

# =============================================================================
# EXTRACTION FUNCTION
# =============================================================================

def extract_article(row):

    prompt = USER_PROMPT.format(
        article_id=row.name,
        publication_date=row["date"],
        source=row["archive"],
        title=row["title"],
        summary=row.get("summary", ""),
        text=row["text"],
    )

    response = client.chat.completions.create(

        model=MODEL,

        temperature=0,

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        response_format={
            "type": "json_schema",
            "json_schema": SOLAR_GRAPH_SCHEMA
        }

    )

    return json.loads(response.choices[0].message.content)

# =============================================================================
# RUN EXTRACTION
# =============================================================================

results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    success = False

    for attempt in range(3):

        try:

            result = extract_article(row)

            results.append(result)

            success = True

            break

        except Exception as e:

            print(f"Row {idx} failed (attempt {attempt+1}): {e}")

            time.sleep(2)

    if not success:

        print(f"Skipping article {idx}")

    if len(results) % SAVE_CHECKPOINT_EVERY == 0:

        with open(OUTPUT_JSON, "w") as f:
            json.dump(results, f, indent=2)

# =============================================================================
# SAVE FINAL JSON
# =============================================================================

with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} extracted graphs.")

# =============================================================================
# FLATTEN NODES
# =============================================================================

nodes = []

for article in results:

    for node in article["nodes"]:

        node_copy = node.copy()

        node_copy["article_id"] = article["article_id"]

        nodes.append(node_copy)

nodes_df = pd.DataFrame(nodes)

nodes_df.to_csv(OUTPUT_NODES, index=False)

print(f"Saved {len(nodes_df)} nodes.")

# =============================================================================
# FLATTEN EDGES
# =============================================================================

edges = []

for article in results:

    edges.extend(article["edges"])

edges_df = pd.DataFrame(edges)

edges_df.to_csv(OUTPUT_EDGES, index=False)

print(f"Saved {len(edges_df)} edges.")

# =============================================================================
# OPTIONAL ANALYSIS
# =============================================================================

print(nodes_df.head())

print(edges_df.head())

print(edges_df["relationship"].value_counts())

print(nodes_df["type"].value_counts())

# Example:
#
# company_nodes = nodes_df[nodes_df["type"] == "Company"]
#
# predictions = edges_df[
#     edges_df["relationship"] == "PREDICTS"
# ]
#
# partnerships = edges_df[
#     edges_df["relationship"] == "PARTNERS_WITH"
# ]
#
# unrelated_articles = [
#     r for r in results if r["unrelated"]
# ]
#
# print(len(unrelated_articles))
